# Counterfactual Analysis

In [1]:
import warnings
import joblib
import json
from tqdm import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

from catboost import CatBoostClassifier


from sklearn.metrics import roc_curve, confusion_matrix, roc_auc_score

from pathlib import Path
import shutil
import sys 
sys.path.append('..')  

from module.dataload import DPN_data
import ymlconfig

import dice_ml

%matplotlib inline
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')
np.set_printoptions(precision=3)  # decimal places for outputs from numpy
pd.set_option("display.precision", 3)  # decimal places for outputs from pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Load / Reload Selection Utility Functions

In [2]:
from utils2 import explainability as exp
from utils2 import counterfactuals as cf

----

## Read Config File

In [35]:
config_path = Path(r'experiments')
config_filename =  "bin_cf_final.yml"
config_dict = ymlconfig.load_config(config_path / config_filename)
config = ymlconfig.dict_to_namespace(config_dict)
config_dict


{'experiment': {'summary': 'binary classification - Counterfactual Analysis (Final)',
  'classification_type': 'binary',
  'stage': 'counterfactuals',
  'tag': 'final',
  'verbosity': 1,
  'random_seed': 42},
 'data': {'dataset_path': '../dataset/Sudoscan Working File with Stats.xlsx'},
 'model': {'code': 'catboost', 'name': 'CatBoost'},
 'explainability': {'ksplit_trained_model_results_file': 'binary\\explainability\\catboost\\final\\catboost_ksplit_trained_models.joblib',
  'rundate': '2026-03-18',
  'tag': 'final'},
 'dice': {'method': 'genetic',
  'CFs': 10,
  'repeats': 1,
  'nzfill': 3,
  'heatmap': {'save_every': 15, 'figsize': {'x': 15, 'y': 6.5}}}}

#### Set output directory

In [4]:
outputdir = config_path /  config.experiment.classification_type /  config.experiment.stage / config.model.code / config.experiment.tag 
outputdir.mkdir(parents=True, exist_ok=True)
print(outputdir)

experiments\binary\counterfactuals\catboost\final


#### Copy config file to output directory

In [5]:
source = config_path / config_filename
destination = outputdir / config_filename
shutil.copy(source, destination)

WindowsPath('experiments/binary/counterfactuals/catboost/final/bin_cf_final.yml')

### Load and verify ksplit_trained_model_results_file from Explainability Stage

In [6]:
ksplit_trained_models = joblib.load(config_path / config.explainability.ksplit_trained_model_results_file)
assert ksplit_trained_models['rundate'] == config.explainability.rundate, f"{ksplit_trained_models['rundate']} != {config.explainability.rundate}"
assert ksplit_trained_models['tag'] == config.explainability.tag
print('rundate:', ksplit_trained_models['rundate'])
print('tag:', ksplit_trained_models['tag'])
ksplit_trained_models['summary']

rundate: 2026-03-18
tag: final


,0,1,2,mean,std
youden,0.909,0.727,0.784,0.807,0.076
roc_auc,0.983,0.986,0.944,0.971,0.019


In [7]:
split_results = ksplit_trained_models['results']
split_results[0]

{'model': <catboost.core.CatBoostClassifier at 0x26420fe4dd0>,
 'X_train':      SEX   AGE  SUBJ  DM_DUR  INSULIN  HBA1C  HPN  PAOD  DSLPDMIA  CKD  GBS  \
 0      1  64.0     1     7.0      1.0  15.00    0     0         0    0    0   
 1      0  59.0     1     1.0      0.0   5.60    1     0         0    0    0   
 5      0  20.0     1     2.0      1.0   7.80    0     0         0    0    0   
 6      0  69.0     0     0.0      0.0   8.00    1     0         1    0    0   
 7      0  60.0     0     2.0      0.0   5.80    1     0         0    0    0   
 8      1  62.0     0     0.0      1.0  14.36    0     0         0    0    0   
 9      0  44.0     1    17.0      0.0   7.01    0     0         0    0    0   
 10     0  70.0     0    10.0      0.0   6.40    1     0         0    1    0   
 11     1  61.0     1     4.0      0.0   8.30    0     0         1    1    0   
 12     1  57.0     1     7.0      1.0  13.00    0     0         1    0    0   
 14     0  78.0     1     5.0      0.0   5.00 

## Data Loading

In [8]:
D = DPN_data(config.data.dataset_path)
D.load(classification=config.experiment.classification_type)

dfdpn = D.df
data_cols = dfdpn.drop(D.non_data_cols, axis=1, errors="ignore").columns
X = dfdpn[data_cols]
y = dfdpn['Confirmed_Binary_DPN']
X.shape, y.shape

((190, 40), (190,))

In [9]:
dfXy = pd.concat([X, y], axis=1)
X.shape, y.shape, dfXy.shape

((190, 40), (190,), (190, 41))

## Prepare Explainer

In [10]:
midx = 0
X_test = split_results[midx]['X_test']
y_test = split_results[midx]['y_test']
threshold = split_results[midx]['threshold']
dfXy_test = pd.concat([X_test, y_test], axis=1)
X_test.shape, y_test.shape, dfXy_test.shape, threshold

((64, 40), (64,), (64, 41), 0.6352084424142594)

In [11]:
allfeature_cols = dfXy_test.columns.drop('Confirmed_Binary_DPN').to_list()
continuous_cols = dfXy_test.columns.difference(D.categorical_cols+['Confirmed_Binary_DPN']).to_list()
print('all feature columns:\n', len(allfeature_cols), allfeature_cols)
print('categorical columns:\n', len(D.categorical_cols), D.categorical_cols)
print('continuous_columns:\n', len(continuous_cols), continuous_cols)

all feature columns:
 40 ['SEX', 'AGE', 'SUBJ', 'DM_DUR', 'INSULIN', 'HBA1C', 'HPN', 'PAOD', 'DSLPDMIA', 'CKD', 'GBS', 'DEC_VS', 'DEC_PPS', 'DEC_LTS', 'DEC_AR', 'MNSI', 'SSA_L', 'SSC_L', 'SPSA_L', 'SPSC_L', 'MCV_L', 'DL_L', 'CMAPANK_L', 'CMAPKNE_L', 'FWAVE_L', 'SSA_R', 'SSC_R', 'SPSA_R', 'SPSC_R', 'MCV_R', 'DL_R', 'CMAPANK_R', 'CMAPKNE_R', 'FWAVE_R', 'FEET_MEAN_ESC', 'FEET_PCT_ASYM', 'HAND_MEAN_ESC', 'HAND_PCT_ASYM', 'NS', 'CAS']
categorical columns:
 12 ['SEX', 'SUBJ', 'INSULIN', 'HPN', 'PAOD', 'DSLPDMIA', 'CKD', 'GBS', 'DEC_VS', 'DEC_PPS', 'DEC_LTS', 'DEC_AR']
continuous_columns:
 28 ['AGE', 'CAS', 'CMAPANK_L', 'CMAPANK_R', 'CMAPKNE_L', 'CMAPKNE_R', 'DL_L', 'DL_R', 'DM_DUR', 'FEET_MEAN_ESC', 'FEET_PCT_ASYM', 'FWAVE_L', 'FWAVE_R', 'HAND_MEAN_ESC', 'HAND_PCT_ASYM', 'HBA1C', 'MCV_L', 'MCV_R', 'MNSI', 'NS', 'SPSA_L', 'SPSA_R', 'SPSC_L', 'SPSC_R', 'SSA_L', 'SSA_R', 'SSC_L', 'SSC_R']


In [12]:
X_train = split_results[midx]['X_train']
X_test = split_results[midx]['X_test']

# convert categorical columns in X_train - needed in CatBoost for use in DiCE
X_train[D.categorical_cols] = X_train[D.categorical_cols].astype(str)
X_test[D.categorical_cols] = X_test[D.categorical_cols].astype(str)

best_params = split_results[midx]['best_params']
y_train = split_results[midx]['y_train']
y_test = split_results[midx]['y_test']

# refit model so we can set cat_features (needed in DiCE)
model=  CatBoostClassifier(**best_params, 
                           cat_features=D.categorical_cols, 
                           verbose=0
                           ).fit(X_train, y_train)

## Global Counterfactual Analysis

###  Define Global Permitted Range

In [13]:
global_permitted_range = cf.get_global_permitted_range(dfXy, continuous_cols, verbosity=1)

,min,max
AGE,8.111,100.889
CAS,0.000,58.027
CMAPANK_L,0.000,29.589
CMAPANK_R,0.000,31.434
CMAPKNE_L,0.000,22.551
CMAPKNE_R,0.000,23.008
DL_L,0.000,48.650
DL_R,0.000,16.399
DM_DUR,0.000,37.797
FEET_MEAN_ESC,0.000,106.560


### Setup Explainer Object

In [14]:
wrapped_model = cf.CatBoostWrapper(model, threshold)
cf.test_wrapped_model(model, wrapped_model, X_test, y_test, threshold)


Confusion Matrix at default threshold (0.5):
[[20  0]
 [ 3 41]]
Confusion Matrix at custom threshold (0.6352084424142594):
[[20  0]
 [ 4 40]]
Rows with different predictions at thresholds: 


,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,pred_0.50,pred_0.64,pred_proba
106,1,33.0,1,3.0,1.0,9.0,0,0,0,0,0,0,1,0,1,3.0,4.88,49.2,0.0,0.0,43.2,3.5,13.14,10.75,43.3,2.68,45.3,0.0,0.0,44.0,3.45,9.98,7.18,42.4,76.0,0.0,76.0,12.0,90.0,4.0,1,0,0.524


### Plot Global Importances

In [ ]:
d = dice_ml.Data(dataframe=dfXy_test, # use only the test set
                 continuous_features=continuous_cols,                  
                 categorical_features=D.categorical_cols,
                 permitted_range = global_permitted_range, 
                 outcome_name='Confirmed_Binary_DPN')

m = dice_ml.Model(model=wrapped_model, backend="sklearn", model_type="classifier")
dexp = dice_ml.Dice(d, m, method=config.dice.method)

cf.plot_global_importance(dexp, D, X_test, midx, config, 
                        highlight_features=[], total_CFs=10, 
                        title_suffix="", filename_suffix="", savedir=None)

## Local Counterfactual Analysis

##### Local Permitted Range for Patient Index 40

In [16]:
pidx = 40
query_instance = X[pidx:pidx+1]
instance_permitted_range = cf.get_local_permitted_range(
    dfXy, query_instance, allfeature_cols, D.categorical_cols, continuous_cols, cf.progressive_cols)

,40,min,max
SEX,0.00,NaN,NaN
AGE,47.00,47.000,58.889
SUBJ,1.00,NaN,NaN
DM_DUR,14.00,14.000,21.797
INSULIN,1.00,NaN,NaN
HBA1C,10.33,7.615,13.045
HPN,0.00,NaN,NaN
PAOD,0.00,NaN,NaN
DSLPDMIA,1.00,NaN,NaN
CKD,0.00,NaN,NaN


##### Generate 5 Sample Local Counterfactuals for Patient 40

In [17]:
e1 = cf.generate_sample_local_cf_with_permitted_range(dfXy, dexp, query_instance, instance_permitted_range, config, CFs=3)
e1_cfdf = e1.cf_examples_list[0].final_cfs_df

generating counterfactuals for the CatBoost model


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:01<00:00,  1.03s/it]

Query instance (original outcome : 1)


,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed_Binary_DPN
0,0,47.0,1,14.0,1.0,10.33,0,0,1,0,0,0,0,0,0,7.0,11.05,49.1,3.61,43.5,40.4,3.45,4.85,3.99,48.3,12.14,43.7,5.02,44.0,41.7,3.45,8.07,7.31,47.9,83.0,3.0,87.0,2.0,84.0,17.0,1



Diverse Counterfactual set (new outcome: 0)


,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed_Binary_DPN
0,-,-,-,-,-,10.3,1.0,-,0.0,-,-,1.0,-,1.0,-,-,12.2,44.7,3.6,41.1,42.2,4.0,-,7.05,50.8,12.3,45.1,5.0,44.9,41.9,3.6,11.57,8.28,50.1,-,-,-,4.0,-,-,0.0
0,-,-,0.0,-,0.0,8.8,1.0,-,-,-,-,-,-,-,-,-,11.1,57.1,3.6,54.7,46.4,3.75,-,-,47.4,20.0,55.6,5.0,56.5,48.6,3.7,10.56,8.69,47.0,69.0,1.0,80.0,1.0,-,-,0.0
0,-,56.0,-,-,0.0,10.3,-,-,0.0,-,-,-,-,-,-,-,16.3,52.0,3.6,50.7,46.6,3.75,9.46,6.93,47.0,13.8,50.0,5.0,53.0,43.8,3.2,7.41,5.61,-,71.0,8.0,-,6.0,-,-,0.0


In [18]:
e1_cfdf.head(2)

,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed_Binary_DPN
0,0,47.0,1,14.0,1.0,10.3,1,0,0,0,0,1,0,1,0,7.0,12.2,44.7,3.6,41.1,42.2,4.00,4.85,7.05,50.8,12.3,45.1,5.0,44.9,41.9,3.6,11.57,8.28,50.1,83.0,3.0,87.0,4.0,84.0,17.0,0
0,0,47.0,0,14.0,0.0,8.8,1,0,1,0,0,0,0,0,0,7.0,11.1,57.1,3.6,54.7,46.4,3.75,4.85,3.99,47.4,20.0,55.6,5.0,56.5,48.6,3.7,10.56,8.69,47.0,69.0,1.0,80.0,1.0,84.0,17.0,0


### Borderline & Misclassified Local Counterfactuals

In [42]:
ioi_df, display_cols = cf.get_instances_of_interest(wrapped_model, X_test, y_test, config=config, split_index=midx, threshold=threshold, delta=0.2, savedir=outputdir)

Found 6 misclassified and borderline cases (|p - 0.6352| ≤ 0.2)


,SEX,AGE,SUBJ,DM_DUR,margin,misclassified,pred_proba,pred,actual
4,1,57.0,0,5.0,0.083,False,0.719,1,1
55,0,67.0,1,11.0,0.131,False,0.767,1,1
67,1,56.0,1,0.0,0.449,True,0.187,0,1
106,1,33.0,1,3.0,0.111,True,0.524,0,1
128,0,74.0,1,30.0,0.360,True,0.275,0,1
158,0,43.0,1,2.0,0.598,True,0.037,0,1


In [54]:
ioi_df.loc[4][['actual','pred']]

actual    1
pred      1
Name: 4, dtype: object

### Generate Local Counterfactuals

In [48]:
ioi_df

,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,pred_proba,margin,pred,actual,misclassified
4,1,57.0,0,5.0,1.0,14.4,0,0,0,0,0,1,0,1,1,1.0,4.19,41.90,3.70,38.20,38.5,4.50,8.89,6.88,48.3,5.36,45.5,4.42,39.50,38.3,4.00,10.29,8.82,49.9,54.0,3.0,63.0,0.0,54.0,36.0,0.719,0.083,1,1,False
55,0,67.0,1,11.0,0.0,11.1,1,0,1,0,0,1,1,1,1,4.0,8.74,47.50,4.03,44.50,45.2,4.85,14.13,9.15,49.2,7.24,46.5,7.04,53.10,42.5,4.00,8.42,6.23,50.0,67.0,2.0,49.0,9.0,47.0,32.0,0.767,0.131,1,1,False
67,1,56.0,1,0.0,0.0,5.1,0,0,0,0,0,0,0,0,0,3.0,16.27,52.00,12.16,50.70,46.6,3.75,9.46,6.93,47.0,13.80,50.0,13.48,53.00,43.8,3.20,7.41,5.61,47.7,71.0,8.0,63.0,6.0,60.0,30.0,0.187,0.449,0,1,True
106,1,33.0,1,3.0,1.0,9.0,0,0,0,0,0,0,1,0,1,3.0,4.88,49.20,0.00,0.00,43.2,3.50,13.14,10.75,43.3,2.68,45.3,0.00,0.00,44.0,3.45,9.98,7.18,42.4,76.0,0.0,76.0,12.0,90.0,4.0,0.524,0.111,0,1,True
128,0,74.0,1,30.0,0.0,4.9,1,0,1,0,0,0,1,1,1,4.0,14.40,47.60,11.03,45.50,45.1,3.95,10.29,7.78,46.4,7.45,46.4,11.65,48.50,42.3,3.45,7.71,5.12,46.6,46.0,0.0,60.0,0.0,34.0,30.0,0.275,0.360,0,1,True
158,0,43.0,1,2.0,0.0,5.2,0,0,1,0,0,0,0,1,0,9.0,12.19,1.96,9.78,2.38,47.2,4.15,14.75,8.91,43.3,34.31,57.7,19.70,2.58,49.3,3.45,15.36,9.43,42.9,57.0,6.0,50.0,7.0,70.0,29.0,0.037,0.598,0,1,True


#### Patient Index 40 (Patient 41 in Spreadsheet)

In [21]:
pidx = 40 # dataframe is zero-indexed
query_instance = X[pidx:pidx+1]
df_dcf = cf.generate_diverse_cfs(
    dexp,
    query_instance, 
    total_CFs=10,
    permitted_range=instance_permitted_range,
    features_to_vary=dfXy.columns.drop(['SEX', 'Confirmed_Binary_DPN']).to_list(),
    seeds=[0]
    )

100%|██████████| 1/1 [01:18<00:00, 78.83s/it]


In [22]:
df_dcf.tail(3)

,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed_Binary_DPN
5,0,47.0,0,14.0,0.0,10.0,0,0,0,0,0,0,0,1,1,7.8,14.4,49.2,11.0,58.8,44.2,3.95,4.85,7.4,46.4,7.4,61.7,0.2,48.5,42.3,2.2,7.71,4.0,46.6,67.0,0.0,68.0,0.0,70.0,17.0,0
6,0,56.8,1,20.9,1.0,9.5,1,0,1,0,0,0,0,0,0,8.1,13.2,40.0,10.4,54.9,34.6,5.20,9.30,7.0,53.0,5.5,46.6,3.4,64.0,45.0,4.0,10.00,7.0,45.0,100.0,3.0,70.0,5.0,93.0,20.0,0
7,0,47.0,0,14.0,0.0,7.6,0,1,1,1,0,0,0,0,1,7.0,14.2,29.6,10.9,21.8,42.4,1.50,0.06,7.0,38.3,3.3,24.7,6.0,37.0,37.0,3.0,3.38,7.0,37.0,93.0,2.0,87.0,2.0,98.0,11.0,0


In [23]:
dfXy.columns.__len__()

41

In [59]:
cf.plot_local_cf_heatmap(dfXy, df_dcf, query_instance, 
                      query_idx=pidx, pred=0, actual=1, config=config, split_index=midx,  
                      savedir=outputdir)

diffs.shape:  (8, 41)
batch_ranges:  [(0, 15)]
idx_start, idx_end:  0 15
diff.shape:  (8, 40)
Counterfactual heatmaps saved to catboost_split0_local_cfidx0-idx7.png in experiments\binary\counterfactuals\catboost\final


##### Get Most Changed Features

In [188]:
most_changed_features = cf.get_most_changed_feature(df_dcf, query_instance)
most_changed_features

sparsity                8
SPSA_R                  8
FWAVE_R                 8
FWAVE_L                 8
L2_dist                 8
MCV_L                   8
MCV_R                   8
SPSA_L                  8
DL_R                    8
DL_L                    8
SPSC_L                  8
HBA1C                   8
SPSC_R                  8
SSA_L                   8
Confirmed_Binary_DPN    8
CMAPKNE_R               8
SSA_R                   8
CMAPANK_R               8
SSC_L                   8
SSC_R                   8
L1_dist                 8
HAND_PCT_ASYM           7
FEET_MEAN_ESC           7
CMAPKNE_L               7
INSULIN                 6
FEET_PCT_ASYM           5
NS                      5
SUBJ                    4
DSLPDMIA                4
MNSI                    4
HAND_MEAN_ESC           4
CAS                     3
DEC_AR                  3
CKD                     3
CMAPANK_L               3
AGE                     3
HPN                     3
DEC_LTS                 2
DM_DUR      

##### Analyze Sparsity and L1, L2 Distances

In [71]:
diffs, cf_ana = cf.analyze_local_cf(query_instance, df_dcf, sort_by="L2_dist")
diffs.head()

,AGE,CAS,CKD,CMAPANK_L,CMAPANK_R,CMAPKNE_L,CMAPKNE_R,Confirmed_Binary_DPN,DEC_AR,DEC_LTS,DEC_PPS,DEC_VS,DL_L,DL_R,DM_DUR,DSLPDMIA,FEET_MEAN_ESC,FEET_PCT_ASYM,FWAVE_L,FWAVE_R,GBS,HAND_MEAN_ESC,HAND_PCT_ASYM,HBA1C,HPN,INSULIN,MCV_L,MCV_R,MNSI,NS,PAOD,SEX,SPSA_L,SPSA_R,SPSC_L,SPSC_R,SSA_L,SSA_R,SSC_L,SSC_R,SUBJ,sparsity,L1_dist,L2_dist
0,0.0,0.0,0.0,0.00,3.50,3.06,0.97,NaN,0.0,1.0,0.0,1.0,0.55,0.15,0.0,-1.0,0.0,0.0,2.5,2.2,0.0,0.0,2.0,-0.03,1.0,0.0,1.8,0.2,0.0,0.0,0.0,0.0,-0.01,-0.02,-2.4,0.9,1.15,0.16,-4.4,1.4,0.0,27,31.40,8.631
1,0.0,0.0,0.0,0.00,2.49,0.00,1.38,NaN,0.0,0.0,0.0,0.0,0.30,0.25,0.0,0.0,-14.0,-2.0,-0.9,-0.9,0.0,-7.0,-1.0,-1.53,1.0,-1.0,6.0,6.9,0.0,0.0,0.0,0.0,-0.01,-0.02,11.2,12.5,0.05,7.86,8.0,11.9,-1.0,28,99.19,29.965
2,9.0,0.0,0.0,4.61,-0.66,2.94,-1.70,NaN,0.0,0.0,0.0,0.0,0.30,-0.25,0.0,-1.0,-12.0,5.0,-1.3,-0.2,0.0,0.0,4.0,-0.03,0.0,-1.0,6.2,2.1,0.0,0.0,0.0,0.0,-0.01,-0.02,7.2,9.0,5.25,1.66,2.9,6.3,0.0,29,84.63,23.605
3,0.0,0.0,1.0,0.00,-0.36,3.41,-3.31,NaN,0.0,0.0,0.0,0.0,0.50,0.25,0.0,-1.0,-16.0,-3.0,-10.0,-0.9,0.0,-7.0,-2.0,-0.33,0.0,-1.0,3.8,6.9,0.8,-15.9,1.0,0.0,-0.01,-4.82,-21.8,12.5,0.05,-8.84,8.0,11.9,-1.0,33,147.38,41.242
4,2.0,6.0,1.0,0.00,-4.69,3.31,0.21,NaN,1.0,0.0,0.0,0.0,0.30,-0.20,6.0,0.0,-12.0,0.0,-0.8,1.2,0.0,0.0,3.0,-2.73,0.0,-1.0,6.2,3.1,1.5,8.0,0.0,0.0,6.99,4.98,8.7,10.7,-4.45,8.86,13.6,4.4,0.0,32,126.92,30.910


#### Filter Valid Counterfactuals

In [171]:
query_instance

,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS
40,0,47.0,1,14.0,1.0,10.33,0,0,1,0,0,0,0,0,0,7.0,11.05,49.1,3.61,43.5,40.4,3.45,4.85,3.99,48.3,12.14,43.7,5.02,44.0,41.7,3.45,8.07,7.31,47.9,83.0,3.0,87.0,2.0,84.0,17.0


In [172]:
list(set(cf.progressive_cols) & set(D.categorical_cols))

['DEC_VS', 'CKD', 'DEC_LTS', 'GBS', 'PAOD', 'DEC_AR', 'HPN', 'DEC_PPS']

In [181]:
df_dcf2 = cf.filter_invalid_progressive_cfs(df_dcf, query_instance, D.categorical_cols)
df_dcf2

Checking for invalid counterfactuals in these columns:
 ['DEC_VS', 'CKD', 'DEC_LTS', 'GBS', 'PAOD', 'DEC_AR', 'HPN', 'DEC_PPS']
All counterfactuals are valid. None was filtered.


,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed_Binary_DPN,sparsity,L1_dist,L2_dist
0,0,47.0,1,14.0,1.0,10.3,1,0,0,0,0,1,0,1,0,7.0,12.2,44.7,3.6,41.1,42.2,4.00,4.85,7.05,50.8,12.3,45.1,5.0,44.9,41.9,3.60,11.57,8.28,50.1,83.0,3.0,87.0,4.0,84.0,17.0,0,27,31.40,8.631
1,0,47.0,0,14.0,0.0,8.8,1,0,1,0,0,0,0,0,0,7.0,11.1,57.1,3.6,54.7,46.4,3.75,4.85,3.99,47.4,20.0,55.6,5.0,56.5,48.6,3.70,10.56,8.69,47.0,69.0,1.0,80.0,1.0,84.0,17.0,0,28,99.19,29.965
2,0,56.0,1,14.0,0.0,10.3,0,0,0,0,0,0,0,0,0,7.0,16.3,52.0,3.6,50.7,46.6,3.75,9.46,6.93,47.0,13.8,50.0,5.0,53.0,43.8,3.20,7.41,5.61,47.7,71.0,8.0,87.0,6.0,84.0,17.0,0,29,84.63,23.605
3,0,47.0,0,14.0,0.0,10.0,0,1,0,1,0,0,0,0,0,7.8,11.1,57.1,3.6,21.7,44.2,3.95,4.85,7.40,38.3,3.3,55.6,0.2,56.5,48.6,3.70,7.71,4.00,47.0,67.0,0.0,80.0,0.0,68.1,17.0,0,33,147.38,41.242
4,0,49.0,1,20.0,0.0,7.6,0,0,1,1,0,0,0,0,1,8.5,6.6,62.7,10.6,52.2,46.6,3.75,4.85,7.30,47.5,21.0,48.1,10.0,54.7,44.8,3.25,3.38,7.52,49.1,71.0,3.0,87.0,5.0,92.0,23.0,0,32,126.92,30.910
5,0,47.0,0,14.0,0.0,10.0,0,0,0,0,0,0,0,1,1,7.8,14.4,49.2,11.0,58.8,44.2,3.95,4.85,7.40,46.4,7.4,61.7,0.2,48.5,42.3,2.20,7.71,4.00,46.6,67.0,0.0,68.0,0.0,70.0,17.0,0,33,134.76,39.576
6,0,56.8,1,20.9,1.0,9.5,1,0,1,0,0,0,0,0,0,8.1,13.2,40.0,10.4,54.9,34.6,5.20,9.30,7.00,53.0,5.5,46.6,3.4,64.0,45.0,4.00,10.00,7.00,45.0,100.0,3.0,70.0,5.0,93.0,20.0,0,32,157.93,40.660
7,0,47.0,0,14.0,0.0,7.6,0,1,1,1,0,0,0,0,1,7.0,14.2,29.6,10.9,21.8,42.4,1.50,0.06,7.00,38.3,3.3,24.7,6.0,37.0,37.0,3.00,3.38,7.00,37.0,93.0,2.0,87.0,2.0,98.0,11.0,0,32,168.99,45.296


In [182]:
# check by manually updating a proressive column from the query instance
query_instance2 = query_instance.copy()
# set progressive column to 1 so we can check 
# if there are counterfactuals that set it to 0 and are thus invalid
query_instance2.HPN = 1 
query_instance2

,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS
40,0,47.0,1,14.0,1.0,10.33,1,0,1,0,0,0,0,0,0,7.0,11.05,49.1,3.61,43.5,40.4,3.45,4.85,3.99,48.3,12.14,43.7,5.02,44.0,41.7,3.45,8.07,7.31,47.9,83.0,3.0,87.0,2.0,84.0,17.0


In [183]:
cf.filter_invalid_progressive_cfs(df_dcf, query_instance2, D.categorical_cols)

Checking for invalid counterfactuals in these columns:
 ['DEC_VS', 'CKD', 'DEC_LTS', 'GBS', 'PAOD', 'DEC_AR', 'HPN', 'DEC_PPS']
Removed 5 invalid progressive counterfactuals


,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed_Binary_DPN,sparsity,L1_dist,L2_dist
0,0,47.0,1,14.0,1.0,10.3,1,0,0,0,0,1,0,1,0,7.0,12.2,44.7,3.6,41.1,42.2,4.00,4.85,7.05,50.8,12.3,45.1,5.0,44.9,41.9,3.6,11.57,8.28,50.1,83.0,3.0,87.0,4.0,84.0,17.0,0,27,31.40,8.631
1,0,47.0,0,14.0,0.0,8.8,1,0,1,0,0,0,0,0,0,7.0,11.1,57.1,3.6,54.7,46.4,3.75,4.85,3.99,47.4,20.0,55.6,5.0,56.5,48.6,3.7,10.56,8.69,47.0,69.0,1.0,80.0,1.0,84.0,17.0,0,28,99.19,29.965
6,0,56.8,1,20.9,1.0,9.5,1,0,1,0,0,0,0,0,0,8.1,13.2,40.0,10.4,54.9,34.6,5.20,9.30,7.00,53.0,5.5,46.6,3.4,64.0,45.0,4.0,10.00,7.00,45.0,100.0,3.0,70.0,5.0,93.0,20.0,0,32,157.93,40.660


#### Sufficiency

A sufficient feature  change is one that can cause the outcome change by itself.


In [ ]:
all_features = dfXy.columns.drop(['SEX', 'Confirmed_Binary_DPN']).to_list(),

most_changed_features_list = most_changed_features.index.to_list()
confirmed_sufficient_features = ['SSA_R', 'SSA_L', 'DL_L']
forced_timeout_features = ['HBA1C',  'DEC_AR', 'DEC_VS', 'DEC_LTS', 'DEC_PPS', 'CMAPKNE_L', 'FEET_PCT_ASYM', 'INSULIN', 'HPN', 'HAND_PCT_ASYM',  'DSLPDMIA', 'DL_R']
# Other exception: 'SPSA_R'
# No CF exception: 'SPSA_L', 'CMAPANK_L', 'MCV_L', 

# set check_features as needed
check_features = confirmed_sufficient_features[:2]
print(check_features)

df_s = cf.check_sufficiency(
    dexp,
    query_instance,
    check_features=check_features,
    maxiterations=5,
    permitted_range=instance_permitted_range,
    )
df_s

#### Necessity

A necessary feature change is one that must be altered; without it, no counterfactual achieves the desired outcome.

In [238]:
from utils2 import counterfactuals as cf
check_features = ['CAS', 'NS']
df_ns = cf.check_necessity(
    dexp,
    query_instance,
    all_features=dfXy.columns.drop(['SEX', 'Confirmed_Binary_DPN']).to_list(),
    maxiterations=500,
    total_CFs=2,
    permitted_range=instance_permitted_range,
    seeds=[0,1,2,3,4], 
    )
df_ns

  0%|          | 0/39 [00:00<?, ?it/s]

100%|██████████| 39/39 [00:58<00:00,  1.49s/it]


,feature,necessary
0,AGE,False
1,SUBJ,False
2,DM_DUR,False
3,INSULIN,False
4,HBA1C,False
5,HPN,False
6,PAOD,False
7,DSLPDMIA,False
8,CKD,False
9,GBS,False


In [220]:
from utils2 import counterfactuals as cf

In [ ]:
# df_ns = check_necessity(
#     dexp,
#     query_instance,
#     all_features=dfXy.columns.drop(['SEX', 'Confirmed_Binary_DPN']).to_list(),
#     maxiterations=500,
#     total_CFs=2,
#     permitted_range=instance_permitted_range,
#     seeds=[0,1,2,3,4], 
#     )

In [ ]:
# from utils2 import counterfactuals as cf
# df_ns = cf.check_necessity(
#     dexp,
#     query_instance,
#     all_features=dfXy.columns.drop(['SEX', 'Confirmed_Binary_DPN']).to_list(),
#     maxiterations=500, 
#     total_CFs=2,
#     permitted_range=instance_permitted_range,
#     nrepeats=5, #5 
#     )
# df_ns

### Generate Files for Counterfactuals by batch

#### Setup

In [ ]:
D = DPN_data("../dataset/Sudoscan Working File with Stats.xlsx")
D.load(classification="binary")
df = D.df
data_cols = df.drop(D.non_data_cols, axis=1, errors="ignore").columns

X = df[data_cols]
y = df['Confirmed_Binary_DPN']
dfXy = pd.concat([X, y], axis=1)

allfeature_cols = dfXy.columns.drop('Confirmed_Binary_DPN').to_list()
continuous_cols = dfXy.columns.difference(D.categorical_cols+['Confirmed_Binary_DPN']).to_list()

actionable_cols = ['HBA1C', 'DSLPDMIA', 'INSULIN']
progressive_cols = ['AGE', 'DM_DUR', 'HPN', 'PAOD', 'CKD', 'GBS', 'DEC_VS', 'DEC_PPS', 'DEC_LTS', 'DEC_AR', 'MNSI']
immutable_cols = ["SEX"]


global_permitted_range = get_global_permitted_range(continuous_cols)

d = dice_ml.Data(dataframe=dfXy, 
                 categorical_features = D.categorical_cols,
                 continuous_features=continuous_cols,                  
                 permitted_range = global_permitted_range,                 
                 outcome_name='Confirmed_Binary_DPN')
m = dice_ml.Model(model=optimized_model, backend="sklearn", model_type="classifier")
exp = dice_ml.Dice(d, m, method="genetic")

In [ ]:
dfb = pd.read_csv(r'outputs\counterfactuals\local\binary\borderline_df_delta0.2.csv', index_col=0)
pindices = dfb.index.to_list()
dfb

In [ ]:
def plot_local_cf_heatmap(df_dcf, query_instance, 
                          pstr, pred, actual, 
                          savedir,
                          save_every=15,  
                          figsize=(15, 6.5)
                          ):   
    # Compute differences (each row in df_large vs. the single row)
    
    diffs = df_dcf - query_instance.iloc[0]
    batch_ranges = [(b*save_every, b*save_every+save_every) for b in range(int(np.ceil(df_dcf.shape[0]/save_every))) ]
    for idx_start, idx_end in batch_ranges:
        print("idx_start, idx_end: ", idx_start, idx_end)
        diff = diffs.iloc[idx_start: idx_end]
        diff = diff[dfXy.drop('Confirmed_Binary_DPN', axis=1).columns]

        # Create mask where values == 0
        mask = diff == 0

        # Plot heatmap
        plt.figure(figsize=figsize)
        ax = sns.heatmap(
            diff,
            mask=mask,            # hide zero differences
            cmap="RdBu",        # diverging color map centered at 0
            center=0,
            annot=True,           # show annotations
            fmt=".2f",            # format annotations to 2 decimal places
            annot_kws={"size": 6},# smaller font
            cbar_kws={'label': 'Difference'}
        )

        ax.set_yticks(np.arange(len(diff)) + 0.5)
        ax.set_yticklabels(diff.index, rotation=0, fontsize=8)

        # ✅ Make x-tick labels smaller too
        ax.set_xticklabels(diff.columns, rotation=45, ha='right', fontsize=8)

        # plt.title("Differences from Instance", fontsize=12)
        plt.title(f"Counterfactuals for Patient {pstr}: predicted {pred}, actual {actual}", fontsize=12, pad=20)
        plt.xlabel("Features")
        plt.ylabel("Counterfactuals")

        # 1. Get the feature values from the query instance
        desired_order = diff.columns.tolist() 
        query_instance_reordered = query_instance[desired_order]

        query_values = query_instance_reordered.iloc[0].values
        NEW_Y_POSITION = -0.05 

        # 2. Loop through each feature to place the value text
        for i, value in enumerate(query_values):
            # i + 0.5 centers the text in the column cell.
            # y = -0.3: Increased separation from the heatmap for visual clarity/alignment.
            ax.text(
                x=i + 0.5,
                y=NEW_Y_POSITION, #-0.3, # Adjusted from -0.2 to -0.3
                s="0" if value==0 else "1" if value==1 else f"{value:.2f}",  # Display the value, formatted
                ha='center',
                va='bottom',
                fontsize=6,
                # fontweight='bold',
                color='#1a1a1a' # Slightly darker color for visibility
            )

        # 3. Add a row header label for the new values
        ax.text(
            x=-0.5, # Position to the left of the Y-axis labels
            y=NEW_Y_POSITION, #-0.3, # Adjusted from -0.2 to -0.3
            s="Instance Values:",
            ha='right',
            va='bottom',
            fontsize=6,
            # fontweight='bold',
            color='#1a1a1a' # Slightly darker color for visibility
        )

        plt.tight_layout()
        os.makedirs(os.path.join(savedir, pstr), exist_ok=True)
        idx_end = min(idx_end, idx_start+diff.shape[0])
        fullfilepath = os.path.join(savedir, pstr, f"local_counterfactual_{pstr}_idx{idx_start}-idx{idx_end-1}.png")
        plt.savefig(fullfilepath)

In [ ]:
import os
def generate_local_cf_reports(X, exp, dfb, pidx, 
                              savedir, 
                              features_to_vary,
                              total_CFs=30,
                              seeds=[0,1,2,3,4]):
    query_instance = X[pidx:pidx+1]
    instance_permitted_range = get_local_permitted_range(query_instance, allfeature_cols, continuous_cols, progressive_cols)

    print('Generating Counterfactuals...')
    df_dcf = generate_diverse_cfs(
        exp,
        query_instance, total_CFs=total_CFs,
        permitted_range=instance_permitted_range,
        features_to_vary=features_to_vary,
        seeds=seeds,
        )
    pstr =str(pidx).zfill(3)
    os.makedirs(os.path.join(savedir, pstr), exist_ok=True)
    df_dcf.to_csv(os.path.join(savedir, pstr, f"local_cf_patient_{pstr}.csv"))

    plot_local_cf_heatmap(df_dcf, query_instance, 
                        pstr=pstr, 
                        pred=dfb.loc[pidx].pred_label, 
                        actual=dfb.loc[pidx].true_label, 
                        savedir=savedir, save_every=15, figsize=(15, 6.5))    
    
    most_changed_features = get_most_changed_feature(df_dcf, query_instance).reset_index()
    most_changed_features.columns = ['feature', 'change count']
    most_changed_features.to_csv(os.path.join(savedir, pstr, f"local_cf_most_changed_patient_{pstr}.csv"))


#### Batch process

In [ ]:
for pidx in pindices: 
    print(f"Generating counterfactual analysis for patient {pidx}")
    generate_local_cf_reports(X, exp, dfb, pidx, 
                              savedir=r"outputs\counterfactuals\local\binary",
                              features_to_vary=dfXy.columns.drop(['SEX', 'Confirmed_Binary_DPN']).to_list(), 
                              total_CFs=30,
                              seeds=[0,1,2,3,4],
                              )
    

### Prototypical and Atypical

##### Prototypical (most representative)

##### Atypical (deviating from standard/common)